# Lab 6：使用 Agentic Document Extraction 构建 AWS 流水线

在本实验中，你将构建一条流水线，把自动化文档处理与一个具备记忆能力和视觉溯源的对话式聊天机器人结合起来。

**学习目标：**
- 在 AWS 上构建事件驱动、无服务器的流水线
- 部署一个 Lambda 函数，使用 LandingAI ADE 自动解析 PDF
- 将解析后的文档导入 Amazon Bedrock Knowledge Base
- 创建一个具备持久记忆和视觉溯源能力的 Strands Agent


## 大纲

**第 1 部分：设置 Lambda 函数**
- [步骤 1：环境设置](#step1)
- [步骤 2：初始化 AWS 客户端](#step2)
- [步骤 3：创建部署包](#step3)
- [步骤 4：创建 IAM 角色](#step4)
- [步骤 5：部署 Lambda 函数](#step5)
- [步骤 6：设置 S3 触发器](#step6)

**第 2 部分：构建知识库**
- [步骤 7：上传待处理文档](#step7)
- [步骤 8：连接到 Bedrock Knowledge Base](#step8)
- [步骤 9：将文档导入知识库](#step9)

**第 3 部分：构建 Agent**
- [步骤 10：创建带视觉溯源的搜索工具](#step10)
- [步骤 11：为 Agent 创建记忆](#step11)
- [步骤 12：创建 Strands Agent](#step12)
- [步骤 13：交互式聊天](#step13)


## 大纲

**第 1 部分：设置 Lambda 函数**
- [步骤 1：环境设置](#step1)
- [步骤 2：初始化 AWS 客户端](#step2)
- [步骤 3：创建部署包](#step3)
- [步骤 4：创建 IAM 角色](#step4)
- [步骤 5：部署 Lambda 函数](#step5)
- [步骤 6：设置 S3 触发器](#step6)

**第 2 部分：构建知识库**
- [步骤 7：上传待处理文档](#step7)
- [步骤 8：连接到 Bedrock Knowledge Base](#step8)
- [步骤 9：将文档导入知识库](#step9)

**第 3 部分：构建 Agent**
- [步骤 10：创建带视觉溯源的搜索工具](#step10)
- [步骤 11：为 Agent 创建记忆](#step11)
- [步骤 12：创建 Strands Agent](#step12)
- [步骤 13：交互式聊天](#step13)


<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> 注意 <b>前置条件 <code>(Prerequisites)</code>：</b> 本实验默认你已经拥有一个 AWS 账户、一个包含 `input/` 与 `output/` 文件夹的 S3 bucket，以及一个连接到输出文件夹的 Bedrock Knowledge Base。用于创建这些资源的链接已在 <code>README</code> 文件中提供。</p>


## 架构概览

完整的数据流如下：

1. **Upload**：用户将 PDF 上传到 S3 的 `input/` 文件夹
2. **Trigger**：S3 自动触发 Lambda 函数
3. **Parse**：Lambda 使用 LandingAI ADE 将 PDF 解析为结构化 markdown
4. **Store**：解析结果（markdown、grounding 数据、chunks）保存到 S3 的 `output/` 文件夹
5. **Index**：Bedrock Knowledge Base 为文档建立索引，以支持语义搜索
6. **Query**：用户向带有记忆能力的 Strands agent 提问

<div align="center">
    <img src="images/architecture_1.png" width="700">
</div>


## 安装所需包

安装 AWS 和 agent 相关的包：
- **boto3**：Python 的 AWS SDK
- **bedrock-agentcore**：用于 agent 的记忆管理
- **strands-agents**：AWS 原生 agent 框架


In [ ]:
# Install required packages
!pip install --quiet boto3 python-dotenv Pillow PyMuPDF bedrock-agentcore strands-agents 

<a id="step1"></a>

## 步骤 1：环境设置

从 `.env` 文件中加载包含 AWS 凭证和配置信息的环境变量。


In [ ]:
import boto3, os, json
from dotenv import load_dotenv

# Load environment variables
_ = load_dotenv()

**`.env` 文件示例：**
```bash
# AWS Credentials
AWS_ACCESS_KEY_ID=your_aws_access_key
AWS_SECRET_ACCESS_KEY=your_aws_secret_key
AWS_REGION=your_aws_region

# S3 Bucket
S3_BUCKET=your_bucket_name

# Bedrock Configuration
BEDROCK_MODEL_ID=your_llm_id
BEDROCK_KB_ID=your_bedrock_knowledge_base_id
DATA_SOURCE_ID=your_data_source_id

# LandingAI ADE API Key
VISION_AGENT_API_KEY=your_vision_api_key
```


<a id="step2"></a>

## 步骤 2：初始化 AWS 客户端

`boto3` 库通过 **client** 连接 AWS 服务。每个 client 都提供对某个特定服务的访问：

| Client | 服务 | 用途 |
|--------|---------|--------|
| `s3_client` | Amazon S3 | 上传/下载文件，管理 bucket |
| `lambda_client` | AWS Lambda | 部署函数，配置触发器 |
| `iam` | IAM | 创建具备权限的角色 |
| `logs` | CloudWatch | 监控 Lambda 执行情况 |
| `bedrock_agent_runtime` | Bedrock | 查询知识库 |
| `bedrock_runtime` | Bedrock | 直接调用 Claude 模型 |


In [ ]:
session = boto3.Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_REGION"),
)

# Create clients
s3_client = session.client("s3")
lambda_client = session.client("lambda")
iam = session.client("iam")  # Add IAM client for Lambda role management
logs = session.client("logs")  # CloudWatch Logs client for monitoring
bedrock_agent_runtime = session.client("bedrock-agent-runtime")
bedrock_runtime = session.client("bedrock-runtime")

# 构建一个具备记忆能力的医疗聊天机器人

这条流水线分为三个部分：

### 第 1 部分：设置 Lambda 函数（步骤 3-5）
打包代码、创建 IAM 角色并部署函数。

<div align="center">
    <img src="images/steps3-5.png" width="700">
</div>

### 第 2 部分：设置触发器（步骤 6）
配置 S3，使其在文件上传时调用 Lambda。

<div align="center">
    <img src="images/step6.png" width="200">
</div>

### 第 3 部分：构建 Agent（步骤 7-12）
上传文档、导入知识库，并创建带记忆能力的 agent。

<div align="center">
    <img src="images/steps7-12.png" width="700">
</div>


### 加载辅助函数

`lambda_helpers.py` 中的辅助函数负责处理 AWS 操作，例如创建部署包、配置 IAM 角色和设置触发器。


In [ ]:
# Import helper functions
import pandas as pd
from lambda_helpers import *

print("Helper functions loaded")

Helper functions loaded


<a id="step3"></a>

## 步骤 3：创建部署包

### 什么是 Lambda 部署包？

AWS Lambda 要求将你的代码和依赖一起打包成一个 **zip 文件**：

```
ade_lambda.zip
├── ade_s3_handler.py          ← 你的代码（handler 函数）
├── landingai_ade/             ← LandingAI ADE 包
│   ├── __init__.py
│   └── ...
├── typing_extensions/         ← typing-extensions 包
│   └── ...
└── boto3/                     ← AWS SDK（通常已预装）
    └── ...
```

辅助函数会通过以下步骤创建该部署包：
1. 创建一个临时目录
2. 将 pip 包安装到该目录
3. 复制你的源代码文件
4. 将所有内容一起打包成 zip


In [ ]:
source_files = ["ade_s3_handler.py"]
requirements = ["landingai-ade", "typing-extensions"]

zip_path = create_deployment_package(
    source_files=source_files,
    requirements=requirements,
    output_zip="ade_lambda.zip",
    package_dir="ade_package"
)

[INFO] Creating deployment package: ade_lambda.zip
   Installing dependencies: landingai-ade, typing-extensions
   Adding source: ade_s3_handler.py
   Creating zip archive...
[OK] Package created: ade_lambda.zip (4.5 MB)


### `ade_s3_handler.py` 文件

这段代码在 Lambda 被触发时运行：

1. **Event received**：S3 上传事件触发 Lambda，并附带文件信息
2. **Validation**：检查它是否为 PDF、不是文件夹、且输出结果尚不存在
3. **Download**：将 PDF 下载到 Lambda 的 `/tmp` 目录
4. **Parse**：ADE API 将 PDF 解析为 markdown 和 chunks
5. **Upload**：结果以三种格式保存到 S3 输出文件夹中：
   - **Markdown**：可读格式的完整文档
   - **Grounding JSON**：包含边界框坐标的所有 chunks
   - **Individual chunks**：每个 chunk 一个文件，用于知识库索引

<div align="center">
    <img src="images/flow_ade_handler.png" width="600">
</div>


### 理解输出文件

当一个 PDF 被处理后，Lambda 会在 S3 的 `output/` 文件夹中生成三类输出：

<div align="center">
    <img src="images/files.png" width="800">
</div>


<a id="step4"></a>


## 步骤 4：创建 IAM 角色

### 什么是 IAM 角色？

Lambda 函数运行在**隔离容器**中，默认并不具备任何权限。**IAM 角色**授予该函数访问特定 AWS 服务的权限。

| 权限 | 用途 |
|------------|--------|
| `s3:GetObject` | 从输入文件夹读取 PDF |
| `s3:PutObject` | 将 markdown 写入输出文件夹 |
| `s3:HeadObject` | 检查输出是否已存在 |
| `logs:CreateLogGroup` | 创建 CloudWatch 日志组 |
| `logs:CreateLogStream` | 为每次执行创建日志流 |
| `logs:PutLogEvents` | 写入日志条目以便调试 |


In [ ]:
role_arn = create_or_update_lambda_role(
    iam_client=iam,
    role_name="lambda-ade-exec-role",
    description="Execution role for LandingAI ADE Lambda"
)

[INFO] Using existing role: lambda-ade-exec-role


<a id="step5"></a>


## 步骤 5：部署 Lambda 函数

部署时需要同时包含以下两个组件：
- **Deployment package**：代码（zip 文件）
- **IAM role**：权限配置

配置内容包括：
- **Environment variables**：运行时可访问的配置
- **Timeout**：900 秒（15 分钟），适用于较大的 PDF
- **Memory**：1024 MB RAM


In [ ]:
env_vars = {
    "VISION_AGENT_API_KEY": os.getenv("VISION_AGENT_API_KEY"),
    "ADE_MODEL": "dpt-2-latest",
    "INPUT_FOLDER": "input/",
    "OUTPUT_FOLDER": "output/",
    "S3_BUCKET": os.getenv("S3_BUCKET"),
    "FORCE_REPROCESS": "false"  # Set to "true" to reprocess all files even if outputs exist
}

response = deploy_lambda_function(
    lambda_client=lambda_client,
    function_name="ade-s3-handler",
    zip_file="ade_lambda.zip",
    role_arn=role_arn,
    handler="ade_s3_handler.ade_handler",
    env_vars=env_vars,
    runtime="python3.10",
    timeout=900,
    memory_size=1024
)

[INFO] Deploying Lambda function: ade-s3-handler
[INFO] Function exists, updating...
   Code updated, waiting for deployment...
[OK] Lambda function updated: ade-s3-handler


<a id="step6"></a>


## 步骤 6：设置 S3 触发器

Lambda 函数虽然已经部署，但还不会自动运行。现在需要配置 S3，使其在**文件上传时触发 Lambda**。

当对象被创建、修改或删除时，S3 都可以发送事件。这里我们将其配置为：当文件上传到 `input/` 文件夹时调用我们的函数。


In [ ]:
# Trigger on all files in input/ folder
setup_s3_trigger(
    s3_client=s3_client,
    lambda_client=lambda_client,
    bucket=os.getenv("S3_BUCKET"),
    prefix="input/",
    function_name="ade-s3-handler",
    suffix=None  # Optional: set to ".pdf" to only trigger on PDF files
)

[INFO] Setting up S3 trigger: s3://sc-landingai/input/ -> ade-s3-handler
   [INFO] Permission may already exist: An error occurred (ResourceConflictException) when calling the AddPermission operation: The statement id (s3invokepermission) provided already exists. Please provide a new statement id, or remove the existing statement.
[OK] S3 trigger set for s3://sc-landingai/input/ -> ade-s3-handler


<a id="step7"></a>


## 步骤 7：上传待处理文档

上传医疗 PDF 文档，并观察这条流水线如何运行。

当你把文件上传到 `input/` 时，这个事件驱动架构会自动：
1. 检测到新文件
2. 触发 Lambda
3. 使用 ADE 解析
4. 将输出保存到 `output/`

<div align="center">
    <img src="images/folders_s3.png" width="750">
</div>


In [ ]:
# Upload medical documents to S3 input folder
local_folder = "medical/"

# Check if folder exists and upload
if os.path.exists(local_folder):
    count = upload_folder_to_s3(
        s3_client=s3_client,
        local_folder=local_folder,
        s3_prefix=f"input/{local_folder}",
        bucket=os.getenv("S3_BUCKET"),
        file_extensions=[".pdf", ".PDF"]
    )
    print(f"\n Waiting for automatic parsing to complete...")
    print("   (The Lambda function will automatically convert PDFs to markdown)")
else:
    print(f" Folder not found: {local_folder}")

[INFO] Uploading medical/ -> s3://sc-landingai/input/medical/
   (Skipping files that already exist in S3)
   [UPLOAD] Uploading: Common_cold_clinincal_evidence.pdf
   [UPLOAD] Uploading: CT_Study_of_the_Common_Cold.pdf
   [UPLOAD] Uploading: Evaluation_of_echinacea_for_the_prevention_and_treatment_of_the_common_cold.pdf
   [UPLOAD] Uploading: Prevention_and_treatment_of_the_common_cold.pdf
   [UPLOAD] Uploading: The_common_cold_a_review_of_the_literature.pdf
   [UPLOAD] Uploading: Understanding_the_symptoms_of_the_common_cold_and_influenza.pdf
   [UPLOAD] Uploading: Viruses_and_Bacteria_in_the_Etiology_of_the_Common_Cold.pdf
   [UPLOAD] Uploading: Vitamin_C_for_Preventing_and_Treating_the_Common_Cold.pdf
[OK] Uploaded 8 files

 Waiting for automatic parsing to complete...
   (The Lambda function will automatically convert PDFs to markdown)


### 监控 Lambda 处理过程

通过 CloudWatch 日志实时监控处理进度：


In [ ]:
stats = monitor_lambda_processing(
    logs_client=logs,
    s3_client=s3_client,
    bucket_name=os.getenv("S3_BUCKET")
)
# To stop monitoring, press Esc followed by double-clicking 'i'

<a id="step8"></a>

## 步骤 8：连接到 Bedrock Knowledge Base

文档已经被解析并存储在 S3 中。下一步是将它们导入 Bedrock Knowledge Base，使其变得**可搜索**。

该知识库已经预先配置为：
- 将 S3 `output/medical_chunks/` 文件夹作为数据源
- 使用 **Amazon Titan** 生成向量嵌入
- 将向量存储在 **OpenSearch Serverless** 中，以支持快速相似度搜索


In [ ]:
# List all your knowledge bases
bedrock_agent = session.client("bedrock-agent")

print("All Knowledge Bases in your account:")
kb_response = bedrock_agent.list_knowledge_bases()

for kb in kb_response.get("knowledgeBaseSummaries", []):
    print(f"\nKnowledge Base: {kb['name']}")
    print(f"   ID: {kb['knowledgeBaseId']}")
    print(f"   Status: {kb['status']}")
    print(f"   Updated: {kb['updatedAt']}")

    # Get data sources for this knowledge base
    ds_response = bedrock_agent.list_data_sources(
        knowledgeBaseId=kb['knowledgeBaseId']
    )

    for ds in ds_response.get("dataSourceSummaries", []):
        print(f"   Data Source: {ds['name']}")
        print(f"      ID: {ds['dataSourceId']}")
        print(f"      Status: {ds['status']}")

All Knowledge Bases in your account:

Knowledge Base: sc-landingai-knowledge-base
   ID: PN252CJ9VO
   Status: ACTIVE
   Updated: 2026-03-15 07:49:09.658250+00:00
   Data Source: sc-landingai-data-source
      ID: AC9YDTU4XV
      Status: AVAILABLE


In [ ]:
BEDROCK_KB_ID = "PN252CJ9VO"
DATA_SOURCE_ID = "AC9YDTU4XV"

<a id="step9"></a>


## 步骤 9：将文档导入知识库

**Ingestion** 会将解析后的文档从 S3 同步到知识库中：

1. 知识库从 S3 读取新增或更新的 JSON 文件
2. 为每个 chunk 创建向量嵌入
3. 将向量存入数据库，以便快速进行相似度搜索

完成后，你就可以使用自然语言进行查询，并检索到相关文档片段。


In [ ]:
response = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=BEDROCK_KB_ID,
    dataSourceId=DATA_SOURCE_ID
)

job_id = response.get("ingestionJob", {}).get("ingestionJobId")
print("✅ Knowledge base sync initiated.")
print(f"   Job ID: {job_id}")

ValidationException: An error occurred (ValidationException) when calling the StartIngestionJob operation: Knowledge base role arn:aws:iam::757370076767:role/service-role/AmazonBedrockExecutionRoleForKnowledgeBase is not able to call specified bedrock embedding model arn:aws:bedrock:us-west-2::foundation-model/amazon.titan-embed-text-v2:0: To access Amazon Bedrock, you must provide further information so we can verify you are a corporate customer and that we can grant you access given applicable law and internal policy. Please submit a request here: https://support.console.aws.amazon.com/support/home#/case/create?issueType=customer-service&serviceCode=account-management&categoryCode=bedrock-allowlisting&locale=en (Service: BedrockRuntime, Status Code: 400, Request ID: 9113b1cc-8862-42dc-9b66-a27e89374619) (SDK Attempt Count: 1)

<a id="step10"></a>


## 步骤 10：创建带视觉溯源的搜索工具

为 agent 创建一个 **search tool**，并加入**视觉溯源**能力，也就是把提取出的信息链接回源文档中的精确位置。

工具流程如下：
1. 使用 **hybrid search**（关键词 + 语义）查询 Bedrock knowledge base
2. 对于 chunk JSON 文件，解析其元数据（chunk_id、page、bbox、type）
3. **生成裁剪后的 chunk 图像**，来源于原始 PDF
4. 将图像上传到 S3，并返回 **presigned URLs**
5. 格式化响应，包含来源、页码、chunk 类型和图像 URL

<div align="left">
    <img src="images/tool.png" width="900">
</div>


In [ ]:
from datetime import datetime
import strands
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
from visual_grounding_helper import (
    extract_chunk_id_from_markdown,
    extract_chunk_image  # Using extract_chunk_image for cropped images
)

@strands.tool
def search_knowledge_base(query: str) -> str:
    """Search the Bedrock knowledge base for relevant medical documents with visual grounding."""
    try:
        # Ensure we have the required environment variables
        kb_id = os.getenv("BEDROCK_KB_ID")  
        bucket = os.getenv("S3_BUCKET")
        if not kb_id:
            return "Error: Knowledge base ID not configured. Please set BEDROCK_KB_ID environment variable."

        # 1. Query the knowledge base using hybrid search 
        
        # Create runtime client if needed
        bedrock_agent_runtime = session.client("bedrock-agent-runtime")
        
        # Query the Knowledge Base with 5 results as requested
        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=kb_id,
            retrievalQuery={"text": query},
            retrievalConfiguration={
                "vectorSearchConfiguration": {
                    "numberOfResults": 5,
                    "overrideSearchType": "HYBRID"  # Use hybrid search for better results
                }
            }
        )
        
        # Get results and sort by score (higher score = more relevant)
        raw_results = response.get("retrievalResults", [])
        sorted_results = sorted(raw_results, key=lambda x: x.get("score", 0), reverse=True)
        
        results = []
        seen_chunk_ids = set()  # Track seen chunk IDs to avoid duplicates

        # 2. For each result, get the location and check if this is a chunk JSON file from medical_chunks folder
        for result in sorted_results:
            content = result.get("content", {}).get("text", "")
            score = result.get("score", 0)
            location = result.get("location", {})
            
            # Get source file from S3 location
            s3_location = location.get("s3Location", {})
            source_uri = s3_location.get("uri", "")
            source_file = source_uri.split("/")[-1] if source_uri else "Unknown source"
            
            # Initialize variables
            chunk_id = None
            visual_info = None
            cropped_image_url = None
            chunk_type = "text"
            page = None
            bbox = None
            source_document = None
            
            # Check if this is a chunk JSON file from medical_chunks folder
            if source_file.endswith('.json') and 'chunks' in source_uri:
                try:
                    # 3. Get chunk data & extract the chunk metadata 
                    # This is a chunk file - parse it directly to get all metadata
                    chunk_key = source_uri.replace(f"s3://{bucket}/", "")
                    chunk_response = s3_client.get_object(Bucket=bucket, Key=chunk_key)
                    chunk_data = json.loads(chunk_response['Body'].read().decode('utf-8'))
                    
                    # Extract all metadata from chunk JSON
                    chunk_id = chunk_data.get('chunk_id', '')
                    chunk_type = chunk_data.get('chunk_type', 'text')
                    page = chunk_data.get('page', 0)
                    bbox = chunk_data.get('bbox', [0, 0, 1, 1])
                    source_document = chunk_data.get('source_document', '')
                    
                    # The text might be in the chunk data or in the content
                    text = chunk_data.get('text', content)
                    
                    # Skip if we've already seen this chunk ID
                    if chunk_id and chunk_id in seen_chunk_ids:
                        continue
                    seen_chunk_ids.add(chunk_id)
                    
                    # 4. Generate cropped chunk image
                    if chunk_id and source_document:
                        source_pdf_key = f"input/medical/{source_document}.pdf"
                        try:
                            s3_client.head_object(Bucket=bucket, Key=source_pdf_key)
                            cropped_image_url = extract_chunk_image(
                                s3_client=s3_client,
                                bucket=bucket,
                                source_pdf_key=source_pdf_key,
                                bbox=bbox,
                                page_num=page,
                                chunk_id=chunk_id,
                                source_document=source_document,
                                highlight=True,
                                padding=10
                            )
                        except:
                            pass  # PDF not found
                            
                except Exception as e:
                    # Fallback if can't parse chunk file
                    pass
            else:
                # Not a chunk file, try to extract chunk ID from markdown
                chunk_id = extract_chunk_id_from_markdown(content)
                
                # Skip if we've already seen this chunk ID
                if chunk_id and chunk_id in seen_chunk_ids:
                    continue
                if chunk_id:
                    seen_chunk_ids.add(chunk_id)
            
            # 5. Format result with all available information
            if cropped_image_url and chunk_id and page is not None:
                # Complete visual grounding available
                result_text = f"""
                **Source:** {source_document or source_file} (Relevance: {score:.2f})
                📄 **Chunk ID:** {chunk_id}
                📍 **Page:** {page}
                🏷️ **Chunk Type:** {chunk_type}
                🔍 **Cropped Chunk Image:** {cropped_image_url}
                
                **Content:**
                {content}"""
                results.append(result_text)
            elif chunk_id and page is not None:
                # Partial visual info (no image but has metadata)
                result_text = f"""
                **Source:** {source_document or source_file} (Relevance: {score:.2f})
                📄 **Chunk ID:** {chunk_id}
                📍 **Page:** {page}
                🏷️ **Chunk Type:** {chunk_type}
                📦 **Bbox:** {bbox if bbox else 'Not available'}
                
                **Content:**
                {content}"""
                results.append(result_text)
            else:
                # No visual grounding available - use content hash as unique ID
                content_hash = hash(content[:200])  # Hash first 200 chars for uniqueness
                if content_hash in seen_chunk_ids:
                    continue
                seen_chunk_ids.add(content_hash)
                
                clean_source = source_file.replace('_grounding.json', '').replace('.json', '').replace('.md', '')
                result_text = f"""**Source:** {clean_source} (Relevance: {score:.2f})
                                **Content:**{content}"""
                results.append(result_text)
        
        if results:
            # Return only top 5 most relevant results with visual references
            return "\n\n---\n\n".join(results[:5])
        else:
            return f"No documents found for query: '{query}'. The knowledge base may be empty or still processing."
            
    except Exception as e:
        error_msg = str(e)
        if "ResourceNotFoundException" in error_msg:
            return f"Error: Knowledge base {kb_id} not found. Please verify the BEDROCK_KB_ID is correct."
        elif "ValidationException" in error_msg:
            return f"Error: Invalid query or configuration. Details: {error_msg}"
        else:
            return f"Error searching knowledge base: {error_msg}"

### 测试搜索函数

在创建 agent 之前，先验证搜索工具是否正常工作：


In [ ]:
# Test the search function before creating agent
print("Testing knowledge base search function...")
test_result = search_knowledge_base("common cold symptoms")
print(f"Test result: {test_result[:200]}...")

if "Error" in test_result:
    print("\n Knowledge base search is not working. Checking configuration...")
    print(f"Current KB ID: {os.getenv('BEDROCK_KB_ID')}")
    print(f"Current Region: {os.getenv('AWS_REGION')}")
else:
    print("\n✅ Knowledge base search is working!")

Testing knowledge base search function...
Test result: Error: Invalid query or configuration. Details: An error occurred (ValidationException) when calling the Retrieve operation: Invalid input or configuration provided. Check the input and Knowledge Base...

 Knowledge base search is not working. Checking configuration...
Current KB ID: PN252CJ9VO
Current Region: us-west-2


In [ ]:
print(test_result)

Error: Invalid query or configuration. Details: An error occurred (ValidationException) when calling the Retrieve operation: Invalid input or configuration provided. Check the input and Knowledge Base configuration and try your request again.


<a id="step11"></a>


## 步骤 11：为 Agent 创建记忆

AWS Bedrock AgentCore 提供**持久化记忆**，使你的 agent 能记住对话内容，并随着时间推移学习用户偏好。

### 记忆策略

| 策略 | 作用 | 示例 |
|----------|--------------|--------|
| **Summary** | 总结过去的会话 | "上次我们讨论了感冒治疗方法" |
| **User Preference** | 学习用户偏好 | "用户偏好简短回答" |
| **Semantic** | 提取并存储事实 | "用户提到自己有过敏问题" |

记忆会跨会话持续保留，从而实现个性化响应。


In [ ]:
# Initialize memory client
memory_client = MemoryClient(region_name=os.getenv("AWS_REGION", "us-west-2"))

# Try to list existing memories first
try:
    existing_memories = memory_client.gmcp_client.list_memories()
    memory_list = existing_memories.get('memories', [])
    
    # Get all MedicalAgentMemory instances and use the most recent
    medical_memories = [m for m in memory_list if 'MedicalAgentMemory' in m.get('id', '')]
    
    if medical_memories:
        # Sort by creation date and take the most recent
        medical_memories.sort(key=lambda x: x.get('createdAt', ''), reverse=True)
        existing_medical_memory = medical_memories[0]
        MEMORY_ID = existing_medical_memory.get('id')
        print(f"Found {len(medical_memories)} existing MedicalAgentMemory instance(s)")
        print(f" Using most recent memory: {MEMORY_ID}")
        print(f"   Created: {existing_medical_memory.get('createdAt', 'N/A')}")
        print(f"   Status: {existing_medical_memory.get('status', 'N/A')}")
    else:
        # Only create if none exist
        raise Exception("No existing MedicalAgentMemory found, will create new one")
        
except Exception as e:
    # Create new memory only if none exists
    print(f" Creating new memory... (Reason: {e})")
    try:
        # Add timestamp to make name unique
        comprehensive_memory = memory_client.create_memory_and_wait(
            name=f"MedicalAgentMemory_{datetime.now().strftime('%Y%m%d_%H%M%S')}", 
            description="Memory for medical document analysis with user preferences",
            strategies=[
                {
                    "summaryMemoryStrategy": {
                        "name": "SessionSummarizer",
                        "namespaces": ["/summaries/{actorId}/{sessionId}"]
                    }
                },
                {
                    "userPreferenceMemoryStrategy": {
                        "name": "PreferenceLearner",
                        "namespaces": ["/preferences/{actorId}"]
                    }
                },
                {
                    "semanticMemoryStrategy": {
                        "name": "FactExtractor",
                        "namespaces": ["/facts/{actorId}"]
                    }
                }
            ]
        )
        MEMORY_ID = comprehensive_memory.get('id')
        print(f" New memory created: {MEMORY_ID}")
    except Exception as create_error:
        print(f" Could not create memory: {create_error}")
        print("Continuing without memory functionality...")
        MEMORY_ID = None

 Creating new memory... (Reason: No existing MedicalAgentMemory found, will create new one)
 New memory created: MedicalAgentMemory_20260315_170851-PG11QrCLz3


### 配置记忆会话

session manager 需要两个标识符：
- **Actor ID**：谁在使用这个 agent（用于个性化）
- **Session ID**：本次对话的唯一标识符


In [ ]:
# Set up memory configuration if memory exists
if MEMORY_ID:
    ACTOR_ID = f"medical_user_{datetime.now().strftime('%H%M%S')}"
    SESSION_ID = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    print(f"   Actor: {ACTOR_ID}")
    print(f"   Session: {SESSION_ID}")

    # Configure memory
    memory_config = AgentCoreMemoryConfig(
        memory_id=MEMORY_ID,
        session_id=SESSION_ID,
        actor_id=ACTOR_ID
    )

    # Create session manager
    session_manager = AgentCoreMemorySessionManager(
        agentcore_memory_config=memory_config,
        region_name=os.getenv("AWS_REGION", "us-west-2")
    )
else:
    session_manager = None
    print("Agent will run without memory")

   Actor: medical_user_171326
   Session: session_20260315_171326


<a id="step12"></a>


## 步骤 12：创建 Strands Agent

将所有内容整合为一个 **Strands Agent**，其配置包括：
- **Model**：通过 Bedrock 调用的 Claude，作为底层 LLM
- **System prompt**：定义 agent 行为的指令
- **Session manager**：用于保存偏好和历史的记忆
- **Tools**：带有视觉溯源能力的 `search_knowledge_base` 函数


In [ ]:
from strands import Agent

# Create the agent with memory and tools
medical_agent = Agent(
    model=os.getenv("BEDROCK_MODEL_ID"),
    name="Medical Document Analyzer with Memory",
    description="Expert agent for medical documents with conversation memory",
    system_prompt="""
        You are a medical document analysis assistant with memory capabilities and visual grounding support.
        You remember our conversations, user preferences, and important facts.
        
        Your capabilities:
        - Search and analyze medical documents from the knowledge base
        - Provide visual grounding information showing exact locations in documents
        - Display page numbers and bounding box coordinates when available
        - Reference annotated images that highlight specific document regions
        - Remember user preferences and conversation history
        - Provide personalized, contextual responses
        - Learn from interactions to improve future responses
        
        IMPORTANT: When you receive search results that include visual grounding information, you MUST include:
        - Page numbers where information was found
        - Location coordinates showing exact position on the page
        - Annotated image URLs that show highlighted text regions
        
        When search results contain these visual markers, preserve them in your response. Do not summarize away the visual grounding details.
        
        Visual grounding format to preserve:
        - **Page:** [number] - shows which page contains the information
        - **Location:** [coordinates] - shows exact position on the page
        - **Annotated Image:** [URL] - provides visual highlight of the referenced text
        
        You have access to medical documents about common cold, treatments, and symptoms.
        Always provide evidence-based insights from the documents with visual references when available.
        When visual grounding is provided in search results, include it in your response to help users see exactly where information comes from.
        """,
    session_manager=session_manager,
    tools=[search_knowledge_base]
)

print(f"\n✅ Medical agent ready with memory and visual grounding!")
print(f"   Model: {os.getenv('BEDROCK_MODEL_ID')}")
print(f"   Tools: {medical_agent.tool_names}")
print("\nThe agent will now:")
print("   - Remember your preferences and conversation history")
print("   - Show exact locations in documents when available")
print("   - Provide visual grounding with page numbers and coordinates")


✅ Medical agent ready with memory and visual grounding!
   Model: us.anthropic.claude-sonnet-4-5-20250929-v1:0
   Tools: ['search_knowledge_base']

The agent will now:
   - Remember your preferences and conversation history
   - Show exact locations in documents when available
   - Provide visual grounding with page numbers and coordinates


<a id="step13"></a>


## 步骤 13：交互式聊天

你的医疗文档 agent 已经准备就绪！开启一个交互式聊天会话，以便：
- 就医疗文档提出问题
- 查看带有页码和图像 URL 的视觉溯源结果
- 告诉 agent 你的偏好（例如，“我喜欢简短回答”）
- 观察 agent 在后续会话中记住这些偏好


<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 注意
&nbsp; <b>运行结果可能不同：</b> AI 聊天模型生成的输出具有动态性和概率性，因此每次执行的结果都可能不同。如果你的结果与视频中展示的不同，请不要惊讶。</p>


In [ ]:
print("=" * 70)
print("Medical Agent - Interactive Chat with Visual Grounding")
print("=" * 70)
print("\nAsk questions about medicine.")
print("Type 'exit', 'quit', or 'bye' to end the conversation.")
print("=" * 70 + "\n")

conversation_num = 0

while True:
    try:
        user_input = input("\nYou: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ['exit', 'quit', 'bye', 'q']:
            print("\n Ending conversation. Goodbye!")
            break

        conversation_num += 1

        # Display the question prominently
        print("\n" + "─" * 70)
        print(f"Question #{conversation_num} [{datetime.now().strftime('%H:%M:%S')}]")
        print(f"   \"{user_input}\"")
        print("─" * 70)

        print("\nAgent Response:")
        print("   Processing...\n")

        # Get and display the response
        result = medical_agent(user_input)
        print(result)

        print("\n" + "=" * 70)

    except KeyboardInterrupt:
        print("\n\n Conversation interrupted. Goodbye!")
        break
    except Exception as e:
        print(f"\n Error: {e}")
        print("Please try again or type 'exit' to quit.")

Medical Agent - Interactive Chat with Visual Grounding

Ask questions about medicine.
Type 'exit', 'quit', or 'bye' to end the conversation.


──────────────────────────────────────────────────────────────────────
Question #1 [17:14:05]
   "what do you know"
──────────────────────────────────────────────────────────────────────

Agent Response:
   Processing...


 Error: An error occurred (ValidationException) when calling the ConverseStream operation: Access to Anthropic models is not allowed from unsupported countries, regions, or territories. Please refer to https://www.anthropic.com/supported-countries for more information on the countries and regions Anthropic currently supports.
└ Bedrock region: us-west-2
└ Model id: us.anthropic.claude-sonnet-4-5-20250929-v1:0
Please try again or type 'exit' to quit.

 Ending conversation. Goodbye!


## 总结

以下是你在 Lab 6 中构建的内容：

| 组件 | 服务 | 功能 |
|-----------|------------|--------|
| **Storage** | Amazon S3 | 存储原始 PDF 和解析输出 |
| **Trigger** | AWS Lambda | 使用 ADE 进行无服务器文档解析 |
| **Vector DB** | Bedrock Knowledge Base | 对文档进行语义搜索 |
| **Agent** | Strands Agents | 对话式交互界面 |
| **Memory** | AgentCore Memory | 记住偏好和历史 |

你可以按需扩展这条流水线，以处理其他类型的文档、添加更多工具，或与其他 AWS 服务集成。
